# Forced Alignment Core

> Data structures for word-level forced alignment results

In [ ]:
#| default_exp forced_alignment_core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional

## ForcedAlignItem

A single word-level alignment result from a forced alignment model. Each item represents one word with its temporal boundaries in the audio.

Note that forced alignment models typically strip punctuation from the input text. The consuming service is responsible for mapping these stripped words back to the original punctuated text.

In [ ]:
#| export
@dataclass
class ForcedAlignItem:
    """A single word-level alignment result."""
    text: str        # The aligned word (punctuation typically stripped by model)
    start_time: float  # Start time in seconds
    end_time: float    # End time in seconds

## ForcedAlignResult

Standardized output for all forced alignment plugins. Contains the list of word-level alignments and optional metadata about the alignment run.

In [ ]:
#| export
@dataclass
class ForcedAlignResult:
    """Standardized output for all forced alignment plugins."""
    items: List[ForcedAlignItem]                          # Word-level alignments
    metadata: Dict[str, Any] = field(default_factory=dict)  # Plugin-specific metadata

## Testing

In [ ]:
# Test ForcedAlignItem creation
item = ForcedAlignItem(text="November", start_time=1.04, end_time=1.6)
print(f"Item: {item}")
assert item.text == "November"
assert item.start_time == 1.04
assert item.end_time == 1.6

# Test serialization round-trip
d = asdict(item)
assert d == {"text": "November", "start_time": 1.04, "end_time": 1.6}
reconstructed = ForcedAlignItem(**d)
assert reconstructed == item
print(f"Serialization round-trip: OK")

Item: ForcedAlignItem(text='November', start_time=1.04, end_time=1.6)
Serialization round-trip: OK


In [ ]:
# Test ForcedAlignResult creation
items = [
    ForcedAlignItem(text="November", start_time=1.04, end_time=1.6),
    ForcedAlignItem(text="the", start_time=1.6, end_time=1.68),
    ForcedAlignItem(text="10th", start_time=1.76, end_time=2.08),
]

result = ForcedAlignResult(
    items=items,
    metadata={"model_id": "Qwen/Qwen3-ForcedAligner-0.6B", "language": "English"}
)

print(f"Result: {len(result.items)} items")
print(f"Metadata: {result.metadata}")
assert len(result.items) == 3
assert result.items[0].text == "November"
assert result.metadata["model_id"] == "Qwen/Qwen3-ForcedAligner-0.6B"

Result: 3 items
Metadata: {'model_id': 'Qwen/Qwen3-ForcedAligner-0.6B', 'language': 'English'}


In [ ]:
# Test ForcedAlignResult serialization round-trip
d = asdict(result)
assert len(d["items"]) == 3
assert d["items"][0] == {"text": "November", "start_time": 1.04, "end_time": 1.6}

# Reconstruct from dict
reconstructed = ForcedAlignResult(
    items=[ForcedAlignItem(**item_dict) for item_dict in d["items"]],
    metadata=d["metadata"]
)
assert len(reconstructed.items) == 3
assert reconstructed.items[0] == result.items[0]
assert reconstructed.metadata == result.metadata
print("ForcedAlignResult serialization round-trip: OK")

ForcedAlignResult serialization round-trip: OK


In [ ]:
# Test minimal result (empty items, no metadata)
minimal = ForcedAlignResult(items=[])
assert minimal.items == []
assert minimal.metadata == {}
print(f"Minimal result: {minimal}")

Minimal result: ForcedAlignResult(items=[], metadata={})


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()